# Visualização e Perguntas de Negócio

## Objetivo
Responder às 6 perguntas analíticas do enunciado sobre o Star Schema do Gold.

**Convenções deste projeto:**
- Comentários e textos: português (PT-BR)
- Identificadores (variáveis, funções): inglês (EN)
  - Constantes: `UPPER_SNAKE_CASE`
  - Demais variáveis: `lower_snake_case`
- Colunas e nomes de tabela: português, `snake_case`
- Símbolos nos logs: ✅ ⚠️ ❌



In [0]:

# Padronização dos nomes dos schemas e volumes
CATALOG = "movies_medallion"
GOLD_SCHEMA_NAME = "gold"

GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")

CATALOG: movies_medallion
BRONZE_SCHEMA: movies_medallion.bronze
SILVER_SCHEMA: movies_medallion.silver
GOLD_SCHEMA: movies_medallion.gold
LANDING_PATH: /Volumes/movies_medallion/bronze/landing


## Pergunta 1 — Qual é a receita total (em R$) somada de todos os filmes da base?

Soma de `receita_brl` da fato. Filmes sem receita não entram (SUM ignora null).

**Resposta:** R$ 804.349.841.784,23

In [0]:
spark.sql(
    f"CREATE OR REPLACE TEMP VIEW vw_receita_brl AS SELECT receita_brl FROM {GOLD_SCHEMA}.fact_movies_performance"
)

display(spark.sql("""
    SELECT SUM(receita_brl) AS receita_total_brl
    FROM vw_receita_brl
"""))


receita_total_brl
804349841784.23


## Pergunta 2 — Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.

Join fato + dim_movies, ordenado por popularidade decrescente.

| # | Filme | Popularidade |
|---|---|---|
| 1 | Talk To Me | 1458.51 |
| 2 | Fast X | 1175.27 |
| 3 | Sound Of Freedom | 1111.04 |
| 4 | Barbie | 1069.34 |
| 5 | Elemental | 1008.94 |

In [0]:
spark.sql(
    f"CREATE OR REPLACE TEMP VIEW vw_popularidade AS SELECT sk_movie_id, popularidade FROM {GOLD_SCHEMA}.fact_movies_performance"
)

spark.sql(
    f"CREATE OR REPLACE TEMP VIEW vw_nome_filme AS SELECT sk_movie_id, titulo FROM {GOLD_SCHEMA}.dim_movies"
)

display(
    spark.sql("""
    SELECT d.titulo, f.popularidade
    FROM vw_popularidade f
    INNER JOIN vw_nome_filme d ON d.sk_movie_id = f.sk_movie_id
    ORDER BY f.popularidade DESC
    LIMIT 5
""")
)

titulo,popularidade
Talk To Me,1458.51
Fast X,1175.27
Sound Of Freedom,1111.04
Barbie,1069.34
Elemental,1008.94


## Pergunta 3 — Quais os gêneros com maior quantidade de filmes?

Join bridge_movie_genre + dim_genres, agrupado por gênero, ordenado decrescente.

| # | Gênero | Filmes |
|---|---|---|
| 1 | Drama | 32.291 |
| 2 | Documentary | 19.003 |
| 3 | Comedy | 18.628 |
| 4 | Thriller | 10.279 |
| 5 | Horror | 9.732 |
| 6 | Romance | 7.640 |
| 7 | Action | 6.049 |
| 8 | Crime | 4.747 |
| 9 | Animation | 4.473 |
| 10 | TV Movie | 4.080 |
| 11 | Science Fiction | 3.769 |
| 12 | Family | 3.723 |
| 13 | Mystery | 3.318 |
| 14 | Fantasy | 3.279 |
| 15 | Adventure | 2.871 |
| 16 | Music | 2.795 |
| 17 | History | 2.417 |
| 18 | War | 960 |
| 19 | Western | 410 |

In [0]:
spark.sql(
    f"CREATE OR REPLACE TEMP VIEW vw_bridge_movie_genre AS SELECT * FROM {GOLD_SCHEMA}.bridge_movie_genre"
)

spark.sql(
    f"CREATE OR REPLACE TEMP VIEW vw_genres AS SELECT * FROM {GOLD_SCHEMA}.dim_genres"
)

display(
    spark.sql("""
        SELECT g.nome_genero AS genero, COUNT(b.sk_movie_id) AS numero_de_filmes
        FROM vw_bridge_movie_genre b
        INNER JOIN vw_genres g ON b.sk_genre_id = g.sk_genre_id
        GROUP BY g.nome_genero
        ORDER BY Numero_de_Filmes DESC
    """)
)

Genero,Numero_de_Filmes
Drama,32291
Documentary,19003
Comedy,18628
Thriller,10279
Horror,9732
Romance,7640
Action,6049
Crime,4747
Animation,4473
TV Movie,4080


## Pergunta 4 — Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking (RANK()).

`RANK() OVER (ORDER BY receita_usd DESC)`. Ranking em USD — referência internacional do setor.

| # | Título | Receita US$ | Receita R$ |
|---|---|---|---|
| 1 | Avengers: Endgame | 2.800.000.000 | 13.913.760.000,00 |
| 2 | Avatar: The Way Of Water | 2.320.250.281 | 11.529.787.696,35 |
| 3 | Avengers: Infinity War | 2.052.415.039 | 10.198.860.811,80 |
| 4 | Spider-man: No Way Home | 1.921.847.111 | 9.550.042.663,98 |
| 5 | The Lion King | 1.663.075.401 | 8.264.154.282,65 |
| 6 | Top Gun: Maverick | 1.488.732.821 | 7.397.811.134,11 |
| 7 | Barbie | 1.428.545.028 | 7.098.725.953,14 |
| 8 | The Super Mario Bros. Movie | 1.355.725.263 | 6.736.869.976,90 |
| 9 | Black Panther | 1.349.926.083 | 6.708.052.691,64 |
| 10 | Star Wars: The Last Jedi | 1.332.698.830 | 6.622.447.026,04 |

In [0]:
spark.sql(
    f"CREATE OR REPLACE TEMP VIEW vw_receitas AS SELECT sk_movie_id, receita_usd, receita_brl FROM {GOLD_SCHEMA}.fact_movies_performance"
)

spark.sql(
    f"CREATE OR REPLACE TEMP VIEW vw_dim_movies_title AS SELECT sk_movie_id, titulo FROM {GOLD_SCHEMA}.dim_movies"
)

display(
    spark.sql("""
        SELECT 
            m.titulo,
            r.receita_usd,
            r.receita_brl,
            RANK() OVER (ORDER BY r.receita_usd DESC) AS posicao_ranking
        FROM vw_receitas r
        INNER JOIN vw_dim_movies_title m ON r.sk_movie_id = m.sk_movie_id
        ORDER BY posicao_ranking
        LIMIT 10
    """)
)

titulo,receita_usd,receita_brl,posicao_ranking
Avengers: Endgame,2800000000.00,13913760000.00,1
Avatar: The Way Of Water,2320250281.00,11529787696.35,2
Avengers: Infinity War,2052415039.00,10198860811.80,3
Spider-man: No Way Home,1921847111.00,9550042663.98,4
The Lion King,1663075401.00,8264154282.65,5
Top Gun: Maverick,1488732821.00,7397811134.11,6
Barbie,1428545028.00,7098725953.14,7
The Super Mario Bros. Movie,1355725263.00,6736869976.90,8
Black Panther,1349926083.00,6708052691.64,9
Star Wars: The Last Jedi,1332698830.00,6622447026.04,10


## Pergunta 5 — Qual ator teve a maior quantidade de participações nos filmes dos últimos 2 anos?

Recorte: `[max(data_lancamento) - 2 anos, max(data_lancamento)]`. Filtra `tipo_pessoa = 'Ator'`.
Exclui placeholder "Nenhum". Contagem por `COUNT(DISTINCT titulo)` — a dim tem ~8% de títulos duplicados (mesmo filme com múltiplos `sk_movie_id`).

**Resposta:** Kevin Hart, 8 filmes

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_movies_last_2_years AS
    WITH latest_date AS (
        SELECT MAX(data_lancamento) AS last_date
        FROM {GOLD_SCHEMA}.dim_movies
        WHERE status_filme = 'Lançado'AND data_lancamento <= CURRENT_DATE()
    ),
        
    date_range AS (
        SELECT 
            last_date,
            last_date - INTERVAL 2 YEAR AS first_date
        FROM latest_date
    )
    SELECT d.sk_movie_id, d.ano_lancamento, d.data_lancamento
    FROM {GOLD_SCHEMA}.dim_movies d
    CROSS JOIN date_range
    WHERE d.data_lancamento BETWEEN date_range.first_date AND date_range.last_date
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_actors AS
    SELECT sk_person_id, nome_pessoa
    FROM {GOLD_SCHEMA}.dim_people
    WHERE tipo_pessoa = 'Ator'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_bridge_movie_person AS
    SELECT *
    FROM {GOLD_SCHEMA}.bridge_movie_person
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_dim_movies AS
    SELECT sk_movie_id, titulo
    FROM {GOLD_SCHEMA}.dim_movies
""")

display(
    spark.sql("""
        SELECT a.nome_pessoa, COUNT(DISTINCT d.titulo) AS numero_de_filmes
        FROM vw_movies_last_2_years m
        INNER JOIN vw_bridge_movie_person b ON m.sk_movie_id = b.sk_movie_id
        INNER JOIN vw_actors a ON b.sk_person_id = a.sk_person_id
        INNER JOIN vw_dim_movies d ON m.sk_movie_id = d.sk_movie_id
        WHERE a.nome_pessoa != 'Nenhum'
        GROUP BY a.nome_pessoa
        ORDER BY numero_de_filmes DESC
        LIMIT 1
    """)
)


nome_pessoa,numero_de_filmes
Kevin Hart,8


## Pergunta 6 — Qual a produtora de filmes teve o maior lucro nos últimos 5 anos?

Recorte: `[max(data_lancamento) - 5 anos, max(data_lancamento)]`. Soma `lucro_usd` por produtora via bridge.
Ranking em USD. Co-produções: cada produtora recebe o lucro completo do filme (comportamento esperado do modelo N:N).

**Resposta:** Universal Pictures — US$ 5.772.329.679,00

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_movies_last_5_years AS
    WITH latest_date AS (
        SELECT MAX(data_lancamento) AS last_date
        FROM {GOLD_SCHEMA}.dim_movies
        WHERE status_filme = 'Lançado' AND data_lancamento <= CURRENT_DATE()

    ),
    date_range AS (
        SELECT 
            last_date,
            last_date - INTERVAL 5 YEAR AS first_date
        FROM latest_date
    )
    SELECT d.sk_movie_id, d.ano_lancamento, d.data_lancamento
    FROM {GOLD_SCHEMA}.dim_movies d
    CROSS JOIN date_range
    WHERE d.data_lancamento BETWEEN date_range.first_date AND date_range.last_date
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_companies AS
    SELECT sk_company_id, nome_produtora
    FROM {GOLD_SCHEMA}.dim_companies
""")

spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_fact AS
        SELECT sk_movie_id, lucro_usd, lucro_brl
        FROM {GOLD_SCHEMA}.fact_movies_performance
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_bridge_movie_company AS
    SELECT *
    FROM {GOLD_SCHEMA}.bridge_movie_company
""")

display(
    spark.sql("""
        SELECT 
            a.nome_produtora,
            SUM(f.lucro_usd) AS lucro_em_dolares,
            SUM(f.lucro_brl) AS lucro_em_reais
        FROM vw_movies_last_5_years m
        INNER JOIN vw_bridge_movie_company b ON m.sk_movie_id = b.sk_movie_id
        INNER JOIN vw_companies a ON b.sk_company_id = a.sk_company_id
        INNER JOIN vw_fact f ON m.sk_movie_id = f.sk_movie_id
        GROUP BY a.nome_produtora
        ORDER BY lucro_em_dolares DESC
        LIMIT 10
    """)
)

nome_produtora,lucro_em_dolares,lucro_em_reais
Universal Pictures,5772329679.00,28683860640.89
Marvel Studios,4953462823.00,24614747460.07
Columbia Pictures,3662050755.00,18197462611.75
Pascal Pictures,2701952454.00,13426542134.42
Illumination,2431353473.00,12081881678.03
Paramount,2239394101.00,11127997166.70
Kevin Feige Productions,2138205367.00,10625170109.71
20th Century Studios,2074815383.00,10310172601.21
Lightstorm Entertainment,1860250281.00,9243955696.35
Heyday Films,1490495872.00,7406572087.14


## Diagnóstico — Títulos duplicados em dim_movies
A dim tem ~8% de títulos repetidos (mesmo filme com múltiplos `sk_movie_id`). Investigação que motivou o uso de `COUNT(DISTINCT titulo)` na Q5.

In [0]:
# Sondagens para verificar se existem titulos com vários ids
display(
    spark.sql(f"""
        SELECT titulo, COUNT(*) AS qtd
        FROM {GOLD_SCHEMA}.dim_movies
        GROUP BY titulo
        HAVING COUNT(*) > 1
        ORDER BY qtd DESC
        LIMIT 5
    """)
)

display(
    spark.sql(f"""
        SELECT 
            COUNT(*) AS total_linhas,
            COUNT(DISTINCT titulo) AS titulos_unicos,
            COUNT(*) - COUNT(DISTINCT titulo) AS linhas_duplicadas_por_titulo
        FROM {GOLD_SCHEMA}.dim_movies
    """)
)

titulo,qtd
Emesis Blue,36
Die Hart 2: Die Harter,30
Home,25
Die Hart: Die Harter,24
Spider-man: Lotus,22


total_linhas,titulos_unicos,linhas_duplicadas_por_titulo
97879,89939,7940
